<a href="https://colab.research.google.com/github/havyasir08-collab/Mini-project-password_manager-___/blob/main/moviesentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle


DATA **COLLECTION**

In [3]:
import os
import json

from zipfile import ZipFile
import pandas as pd

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


Data collection - kaggle API

In [4]:
import json
import os
from google.colab import files

print("Please upload your 'kaggle.json' file:")
uploaded = files.upload()

if uploaded:
    # Grab the actual filename Colab saved it as (e.g., 'kaggle.json' or 'kaggle (2).json')
    uploaded_filename = next(iter(uploaded))
    file_bytes = uploaded[uploaded_filename]

    # Load into dictionary
    kaggle_dictionary = json.loads(file_bytes.decode('utf-8'))

    # Save directly to ~/.kaggle/kaggle.json
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_path = os.path.join(kaggle_dir, 'kaggle.json')

    with open(kaggle_path, 'wb') as f:
        f.write(file_bytes)

    os.chmod(kaggle_path, 0o600)

    # Clean up the duplicate file from /content/
    if os.path.exists(uploaded_filename):
        os.remove(uploaded_filename)

    print(f"Successfully loaded '{uploaded_filename}' and configured Kaggle credentials!")
else:
    print("Error: No file was uploaded.")

Please upload your 'kaggle.json' file:


Saving kaggle (1).json to kaggle (1).json
Successfully loaded 'kaggle (1).json' and configured Kaggle credentials!


In [5]:
# setup kaggle credentials as environment variable
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']

In [6]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews --unzip

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:02<00:00, 10.9MB/s]



In [7]:
!ls

'IMDB Dataset.csv'   sample_data


In [8]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:02<00:00, 10.4MB/s]



In [9]:
from zipfile import ZipFile

with ZipFile('/content/imdb-dataset-of-50k-movie-reviews.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [10]:
import pandas as pd

data = pd.read_csv("/content/IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
data.shape

(50000, 2)

In [12]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [14]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [15]:
data.replace({"sentiment":{"positive":1, "negative": 0}}, inplace=True)

/tmp/ipykernel_475/900931923.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment":{"positive":1, "negative": 0}}, inplace=True)


In [16]:
from sklearn.model_selection import train_test_split

# split data into train data and test data
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [17]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


data **preproccesing**

In [18]:
#tokenize text data
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

#tokenize text data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [19]:
print(X_train)
print(X_test)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]
[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [20]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [21]:
print(Y_train)


39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


LSTM - LONG SHORT TERM **MEMORY**

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense



In [23]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

model.build(input_shape=(None, 200))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
#compile the model
model.compile(optimizer="adam", loss="binary_crossentropy",metrics=["accuracy"])

# trainning the  **model**

In [ ]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 222s 428ms/step - accuracy: 0.7956 - loss: 0.4495 - val_accuracy: 0.8505 - val_loss: 0.3458
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 213s 425ms/step - accuracy: 0.8568 - loss: 0.3482 - val_accuracy: 0.8547 - val_loss: 0.3405
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 217s 434ms/step - accuracy: 0.8840 - loss: 0.2907 - val_accuracy: 0.8687 - val_loss: 0.3241
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 215s 430ms/step - accuracy: 0.8966 - loss: 0.2640 - val_accuracy: 0.8695 - val_loss: 0.3261
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 397ms/step - accuracy: 0.9100 - loss: 0.2291

MODE **EVALUATION**

In [26]:
# Corrected syntax
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 132ms/step - accuracy: 0.8790 - loss: 0.3009
Test Loss: 0.3008860647678375
Test Accuracy: 0.8790000081062317


building a predictive sysytem

In [28]:
def predict_sentiment(review):
   # tokenizer and pad the review
   sequence = tokenizer.texts_to_sequences(review)
   padded_sequence = pad_sequences(sequence, maxlen=200)
   prediction = model.predict(padded_sequence)
   sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
   return sentiment

In [29]:
# example usage
new_review =  "the movie was fantastic. i love it"
sentiment = predict_sentiment([new_review])
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step
The sentiment of the review is: positive


In [30]:
new_review = "the movie was bad. i hate it"
sentiment = predict_sentiment([new_review])
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
The sentiment of the review is: negative
